<a href="https://colab.research.google.com/github/tdog9237/swot/blob/master/copy_of_untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

Exception in thread Thread-4 (run_ollama_serve):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-1-2856519456.py", line 6, in run_ollama_serve
  File "/usr/lib/python3.11/subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/usr/lib/python3.11/subprocess.py", line 1955, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: 'ollama'


In [2]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

Exception in thread Thread-9 (run_ollama_serve):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-2-2856519456.py", line 6, in run_ollama_serve
  File "/usr/lib/python3.11/subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/usr/lib/python3.11/subprocess.py", line 1955, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: 'ollama'


In [3]:
xtts_api_server --host 0.0.0.0 --deepspeed


SyntaxError: invalid syntax (ipython-input-3-2477631159.py, line 1)

In [ ]:
from flask import Flask, request, send_file
import os
import requests
from datetime import datetime
from pydub import AudioSegment

app = Flask(__name__)

VOICE_CACHE_DIR = "voice_cache"
XTTS_URL = "http://localhost:8020/tts_to_file"
OLLAMA_URL = "http://localhost:11434/api/generate"
SPEAKER_WAV = "freeman_combined.wav"
LANGUAGE = "en"
PRESET = "low"
PITCH = -2

os.makedirs(VOICE_CACHE_DIR, exist_ok=True)

def summarize_text(text):
    try:
        payload = {
            "model": "nous-hermes",
            "prompt": f"Summarise the following in four sentences max:\n{text}",
            "stream": False
        }
        response = requests.post(OLLAMA_URL, json=payload)
        response.raise_for_status()
        output = response.json()
        return output.get("response", "Sorry, I couldn't process that right now.")
    except Exception as e:
        print(f"[ERROR] Ollama API call failed: {e}")
        return "Sorry, I couldn't process that right now."

def convert_wav_to_pcm(path):
    try:
        sound = AudioSegment.from_file(path)
        # Convert to 16-bit PCM, mono, 44.1kHz (standard WAV)
        sound = sound.set_frame_rate(44100).set_channels(1).set_sample_width(2)
        sound.export(path, format="wav")
        print("[DEBUG] Converted WAV to standard PCM format.")
    except Exception as e:
        print(f"[ERROR] Could not convert WAV: {e}")

def generate_tts_audio(text, output_path):
    try:
        payload = {
            "text": text,
            "speaker_wav": SPEAKER_WAV,
            "language": LANGUAGE,
            "file_name_or_path": output_path  # use absolute path here
        }
        # Send JSON, not multipart
        response = requests.post(XTTS_URL, json=payload)
        response.raise_for_status()

        # According to your XTTS logs, it returns JSON with output_path, not audio bytes.
        # So we expect the file to be written by the service.
        if not os.path.exists(output_path):
            print(f"[ERROR] Expected TTS file not found at {output_path}")
            return None

        # Convert to standard PCM WAV just in case
        convert_wav_to_pcm(output_path)

        return output_path
    except Exception as e:
        print(f"[ERROR] XTTS error: {e}")
        return None

@app.route("/process_text", methods=["POST"])
def process_text():
    try:
        data = request.get_json()
        prompt = data.get("prompt", "")
        print(f"🧠 Prompt received: {prompt}")

        summarized = summarize_text(prompt)
        print(f"💬 Summarized response: {summarized}")

        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        output_wav = os.path.abspath(os.path.join(VOICE_CACHE_DIR, f"tts_{timestamp}.wav")).replace("\\", "/")

        wav_file = generate_tts_audio(summarized, output_wav)

        if wav_file:
            return send_file(wav_file, mimetype="audio/wav")
        else:
            return "TTS generation failed", 500
    except Exception as e:
        print(f"[ERROR] Server processing error: {e}")
        return "Internal server error", 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=True)



 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with stat
